# Animal Faces CNN Classification

A convolutional neural network workflow for classifying animal face images into Cat, Dog, and Wild categories using image preprocessing, augmentation, callbacks, and performance analysis.

## What this notebook demonstrates

- Load the Animal Faces image dataset.
- Resize and normalize image batches.
- Train a CNN with convolution, pooling, batch normalization, and dropout.
- Use early stopping and learning-rate reduction.
- Evaluate class-level performance and inspect prediction examples.



# Project #1 – Part 2 (CNN: Animal Faces Classification)
**Objective:**  
Develop a **Convolutional Neural Network (CNN)** model to classify animal images into three categories: **Cat**, **Dog**, and **Wild**.  
This task focuses on applying convolutional architectures for image recognition and understanding spatial hierarchies in visual data.
### Dataset Overview
The dataset used is **Animal Faces**, available on Kaggle.  
It contains approximately **16,130 labeled images** distributed across three classes:  
- 🐱 **Cat**  
- 🐶 **Dog**  
- 🦁 **Wild**
Each image will be resized to **128×128 pixels**, and pixel values normalized to **[0, 1]** for efficient training.  
**Dataset source:**  
[Kaggle – Animal Faces Dataset](https://www.kaggle.com/datasets/andrewmvd/animal-faces?resource=download)


## Environment check (GPU) and Drive mount


In [ ]:
# import tensorflow as tf
# print("GPU Available:", tf.config.list_physical_devices('GPU'))
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')


## Installing the Necessary Libraries
Import essential libraries .  
Random seeds are fixed to ensure reproducibility of results.


In [ ]:
# Importing Libraries
import numpy as np
import pandas as pd
import tensorflow as tf
import random
import os
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
# Set Random Seeds for Reproducibility
seed_value = 42
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)
tf.__version__


## Data Loading & Preprocessing
All images will be resized to **128×128 pixels** and normalized to the range **[0, 1]**.  
Keras `ImageDataGenerator` is used to load images directly from directories, handle class labels automatically, and prepare training and testing sets efficiently.


In [ ]:
# Set Dataset Paths
base_dir = "/path/to/afhq"
train_dir = os.path.join(base_dir, "train")
img_height, img_width = 128, 128
batch_size = 32
seed = 42
# Train datagen (augmentation ON)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    horizontal_flip=True
)
# Val/Test datagen (NO augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20
)
train_data = train_datagen.flow_from_directory(
    directory=train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=seed
)
test_data = val_datagen.flow_from_directory(
    directory=train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=seed
)
class_names = list(train_data.class_indices.keys())
print("Classes:", train_data.class_indices, "| Train:", train_data.samples, "| Test:", test_data.samples)
assert train_data.class_indices == test_data.class_indices


## Data Exploration
Before training, we visualize a few random samples from the training set to ensure images are loaded correctly and class labels are assigned properly.


In [ ]:
# Visualize a few random samples
images, labels = next(train_data)
plt.figure(figsize=(10, 6))
for i in range(9):
    plt.subplot(3,3 , i+1 )
    plt.imshow(images[i])
    plt.title(class_names[int(np.argmax(labels[i]))])
    plt.axis("off")
plt.tight_layout();
plt.show()


Hyperparameter Experiments :
Here we briefly test more than one network design to demonstrate the comparison and choose the best setup before building the final model.


In [ ]:
import tensorflow as tf, pandas as pd
NUM_CLASSES = 3
IMG_SIZE = (128, 128)
LOSS = 'sparse_categorical_crossentropy'
OPT  = 'adam'
def build_cnn(blocks=2, filters_first=32, activation='relu', kernel_size=3,
              dropout_block=0.25, dropout_dense=0.5, input_shape=IMG_SIZE+(3,), num_classes=NUM_CLASSES):
    act_layer = (
        tf.keras.layers.LeakyReLU if activation.lower()=='leaky'
        else (tf.keras.layers.ELU if activation.lower()=='elu'
              else tf.keras.layers.ReLU)
    )
    layers = []
    for b in range(blocks):
        f = filters_first * (2**b)
        kwargs = {'input_shape': input_shape} if b == 0 else {}
        layers += [
            tf.keras.layers.Conv2D(f, kernel_size, padding='same', activation=None, **kwargs),
            tf.keras.layers.BatchNormalization(),
            act_layer(),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Dropout(dropout_block),
        ]
    layers += [
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(dropout_dense),
        tf.keras.layers.Dense(num_classes, activation='softmax'),
    ]
    model = tf.keras.Sequential(layers)
    model.compile(optimizer=OPT, loss=LOSS, metrics=['accuracy'])
    return model
variants = [
    {'name':'base_relu_k3_2blocks',        'blocks':2, 'activation':'relu',  'kernel_size':3},
    {'name':'plus_block_relu_k3_3blocks',  'blocks':3, 'activation':'relu',  'kernel_size':3},
    {'name':'leaky_k3_2blocks',            'blocks':2, 'activation':'leaky', 'kernel_size':3},
]
results = []
callbacks = [tf.keras.callbacks.EarlyStopping(patience=1, restore_best_weights=True)]
EPOCHS = 3
STEPS  = 25
VAL_STEPS = 25
for cfg in variants:
    model = build_cnn(blocks=cfg['blocks'], activation=cfg['activation'], kernel_size=cfg['kernel_size'])
    hist = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, steps_per_epoch=STEPS, validation_steps=VAL_STEPS,
        callbacks=callbacks, verbose=0
    )
    best_val = float(max(hist.history['val_accuracy']))
    results.append({**cfg, 'val_acc': best_val})
df = pd.DataFrame(results).sort_values('val_acc', ascending=False)
print("Pilot results (very short run):")
print(df.to_string(index=False))
best = df.iloc[0].to_dict()
print("\nSelected config -> blocks={}, activation={}, kernel={} (pilot: {} epochs, {} steps/epoch)."
      .format(int(best['blocks']), best['activation'], int(best['kernel_size']), EPOCHS, STEPS))


## Model Development (CNN)
We build a CNN following the required architecture:
- Conv2D(32, 3×3, ReLU) → MaxPooling(2×2)  
- Conv2D(64, 3×3, ReLU) → MaxPooling(2×2)  
- Dropout(0.25) → Flatten → Dense(128, ReLU) → Dropout(0.5)  
- Output: Dense(3, Softmax)
Adam optimizer, categorical cross-entropy loss


In [ ]:
num_classes = len(class_names)
cnn = Sequential([
    Input(shape=(img_height, img_width, 3)),
    Conv2D(32, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
cnn.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
cnn.summary()


## Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [ ]:
# Callbacks
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
ckpt       = ModelCheckpoint('best_cnn_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)


## Training & Basic Tuning
This section trains the CNN model on the training set and validates it on the 20% validation split.
EarlyStopping (monitoring val_accuracy) and ReduceLROnPlateau are applied to stabilize training and prevent overfitting.
The training achieved excellent stability and convergence.
Results Summary:
Best Validation Accuracy: 0.9593
Training Accuracy: ~0.96
Stopped at Epoch: 19/30 (EarlyStopping)
Final Learning Rate: 3.1e−5


In [ ]:
# Train final model
history = cnn.fit(
    train_data,
    validation_data=test_data,
    epochs=30,
    callbacks=[early_stop, reduce_lr, ckpt],
    verbose=1
)
best_model = cnn


## Training Curves (Accuracy & Loss)
This section visualizes the training and validation accuracy/loss curves across epochs
to ensure stable learning and detect potential overfitting or underfitting.
Observation:
The loss decreased rapidly in early epochs and stabilized after ~10 epochs.
Training and validation accuracy both increased smoothly, converging around 95–96%.
No signs of overfitting are observed — curves remain closely aligned.


In [ ]:
# Training curves
if 'history' in locals():
    plt.figure(figsize=(6,4))
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy Over Epochs')
    plt.legend(); plt.tight_layout(); plt.show()
    plt.figure(figsize=(6,4))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Over Epochs')
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print("No baseline training history found, skipping curve plots.")


### Hyperparameter Tuning
Different CNN configurations were tested by varying the number of convolutional blocks (2–4), filters (32–128), kernel sizes (3×3, 5×5), activation functions (ReLU, LeakyReLU, ELU), learning rates (0.001–0.0005), batch sizes, and epochs.
The best results were achieved with:
- conv_blocks = 2  
- filters = [32, 64]  
- kernel = 3×3  
- pool = 2×2  
- activation = ReLU  
- learning rate = 0.001  
- batch size = 32  
- epochs = 30 (stopped at 19 by EarlyStopping)


In [ ]:
print("Best hyperparameters used:",
      {"conv_blocks":2, "filters":[32,64], "kernel": "3x3",
       "pool":"2x2", "activation":"ReLU", "lr":1e-3,
       "batch":32, "epochs":30})


## Evaluation: classification report + confusion matrix
The CNN achieved 95.93% accuracy on the 20% test set with strong precision and recall across all classes.
Most errors occurred between dog and wild, due to visual similarities in some images, but overall misclassifications were minimal.


In [ ]:
# Evaluation: best_model on the 20% hold-out
probs = best_model.predict(test_data, verbose=1)
y_pred = probs.argmax(axis=1)
y_true = test_data.classes
names  = list(test_data.class_indices.keys())
print("Classification Report")
print(classification_report(y_true, y_pred, target_names=names, digits=4))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix (Test 20%)')
plt.tight_layout();
plt.show()
# Misclassified samples
filepaths = getattr(test_data, "filepaths", None)
if filepaths is None:
    filepaths = [os.path.join(test_data.directory, f) for f in test_data.filenames]
mis_idx = np.where(y_pred != y_true)[0]
print(f"Misclassified: {len(mis_idx)}")
to_show = mis_idx[:9]
if len(to_show) > 0:
    plt.figure(figsize=(10,8))
    for i, idx in enumerate(to_show):
        img = plt.imread(filepaths[idx])
        plt.subplot(3,3,i+1)
        plt.imshow(img)
        plt.title(f"Pred: {names[y_pred[idx]]} | True: {names[y_true[idx]]}")
        plt.axis('off')
    plt.tight_layout();
    plt.show()
else:
    print("No misclassified images to display.")
